# Latent Cartographer: a 3-minute walkthrough

A 0.8M-parameter GPT was trained only on move tokens (`N E S W`) from random walks in a hidden 8×8 maze.
This notebook reproduces the headline results on a CPU:

1. the model's next-move predictions match an exact Bayesian filter;
2. a linear probe reads a belief over all 64 cells out of its residual stream;
3. steering that belief changes which moves the model thinks are legal;
4. steering to every cell lets us redraw the whole maze from the model alone.

Live demo: https://ps12007.github.io/latent-cartographer/

In [ ]:
# Colab setup (skip if running inside the repo)
import os
if not os.path.exists("cartographer"):
    !git clone -q https://github.com/PS12007/latent-cartographer.git
    %cd latent-cartographer

In [ ]:
import numpy as np, torch, torch.nn.functional as F
import matplotlib.pyplot as plt
from cartographer.env import Maze, BayesFilter, sample_walks, MOVES, UNK
from cartographer.model import load_model
from cartographer.analysis import make_dataset, collect, fit_belief_probe
torch.manual_seed(0)

model, ck = load_model("runs/main/model.pt")
maze = Maze.from_dict(ck["maze"])
print(f"{model.n_params():,} parameters")
print(maze.ascii())

## 1. Behaviour: model vs the exact Bayesian filter
A walk with an unknown start. Watch the optimal distribution and the model's agree as the filter localises.

In [ ]:
rng = np.random.default_rng(3)
tokens, cells, _ = sample_walks(maze, 1, 30, p_known=0.0, rng=rng)
beliefs, opt = BayesFilter(maze).run(torch.tensor(tokens))
with torch.no_grad():
    p_model = F.softmax(model(torch.tensor(tokens)), -1)[0]
for t in [0, 2, 5, 10, 20, 29]:
    cand = int((beliefs[0, t] > 1e-9).sum())
    print(f"after {t:2d} moves | candidate cells {cand:2d} | optimal "
          + " ".join(f"{m}:{p:.2f}" for m, p in zip(MOVES, opt[0, t].tolist()))
          + " | model " + " ".join(f"{m}:{p:.2f}" for m, p in zip(MOVES, p_model[t].tolist())))

## 2. Read the belief out of the residual stream
A 64-way softmax probe on the stream after block 3, fitted to the exact posterior.

In [ ]:
train = make_dataset(maze, 1500, 64, 0.35, seed=101)
X = collect(model, train["tokens"], ["resid_3"])["resid_3"]
probe = fit_belief_probe(X.reshape(-1, X.shape[-1]), torch.tensor(train["belief"]).reshape(-1, maze.N), steps=800)

cache = {}
with torch.no_grad():
    model(torch.tensor(tokens), cache=cache)
    decoded = F.softmax(probe(cache["resid_3"][0]), -1)

ts = [0, 3, 6, 12, 24]
fig, ax = plt.subplots(2, len(ts), figsize=(12, 5))
for j, t in enumerate(ts):
    ax[0, j].imshow(beliefs[0, t].reshape(8, 8), cmap="Blues"); ax[0, j].set_title(f"t={t}")
    ax[1, j].imshow(decoded[t].reshape(8, 8), cmap="Reds")
    for a in ax[:, j]:
        r, c = divmod(int(cells[0, t]), 8); a.plot(c, r, "ko", mfc="none"); a.axis("off")
ax[0, 0].text(-3, 3.5, "exact Bayes", rotation=90, va="center")
ax[1, 0].text(-3, 3.5, "from model", rotation=90, va="center")
plt.show()

## 3. Steer the belief and watch the predictions follow
Move the newest token's residual vector along the probe's direction for another cell.

In [ ]:
loc = train["belief"][:, 40].max(-1) > 0.99
base_tokens = train["tokens"][loc][:200, :41]
base_cells = torch.tensor(train["cell"][loc][:200, 40])
Xb = collect(model, base_tokens, ["resid_3"])["resid_3"][:, -1]
Xall = X.reshape(-1, X.shape[-1]); cell_all = torch.tensor(train["cell"]).reshape(-1)
loc_all = torch.tensor(train["belief"]).reshape(-1, maze.N).max(-1).values > 0.99
means = torch.stack([Xall[loc_all & (cell_all == c)].mean(0) for c in range(maze.N)])
W = probe.lin.weight.detach()

def steer(target, alpha=2.0):
    d = W[target] - W[base_cells]; d = d / d.norm(dim=-1, keepdim=True).clamp_min(1e-9)
    scale = (means[target] - means[base_cells]).norm(dim=-1, keepdim=True)
    delta = alpha * scale * d
    delta[base_cells == target] = 0
    def hook(l, x):
        if l == 3:
            x = x.clone(); x[:, -1] += delta
        return x
    with torch.no_grad():
        return F.softmax(model(torch.tensor(base_tokens), resid_hook=hook)[:, -1], -1)

target = 27
p = steer(target)
legal = torch.tensor(maze.open)
ok = ((p > 0.1) == legal[target]).all(-1).float().mean()
print(f"steered to cell {target}: open sides {[m for m, o in zip(MOVES, maze.open[target]) if o]}")
print(f"model's predicted legal moves match that cell in {ok:.0%} of {len(p)} walks")

## 4. Redraw the maze from the model alone

In [ ]:
recon = np.stack([steer(c).mean(0).numpy() for c in range(maze.N)])   # (64, 4) mean P(move) per steered cell
pred_open = recon > 0.1
hit = tot = 0
for c in range(maze.N):
    r, q = divmod(c, 8)
    for m, nb, ok_ in ((1, c + 1, q < 7), (2, c + 8, r < 7)):
        if ok_:
            tot += 1
            hit += int((0.5 * (recon[c, m] + recon[nb, (m + 2) % 4]) > 0.1) == maze.open[c, m])
print(f"{hit}/{tot} interior walls recovered correctly")
print(Maze(8, 8, pred_open).ascii())